# Regresión Lineal para Predecir el LTV a 24 Meses

**Objetivo de aprendizaje:**
Descubrir variables que impactan el KPI de LTV a 24 meses mediante una regresión multivariable, e interpretar el modelo para apoyar la toma de decisiones.

### Contexto del caso: Tlacuachitos Express

Tlacuachitos Express es un servicio ficticio de entregas que desea predecir el valor que generará un cliente en los próximos 24 meses. El área de retención quiere utilizar este modelo para segmentar a los clientes y priorizar esfuerzos de fidelización.

### Exploración del KPI: ¿Qué es el LTV?

Discusión: ¿Qué es el Lifetime Value (LTV)? ¿Por qué es un KPI clave?

Definición del KPI: Suma de todas las compras realizadas por un cliente en sus primeros 24 meses.

## Pasos

### 1. Colección y comprensión de datos

In [1]:
import pandas as pd

df_data = pd.read_csv('./resources/tlacuachitos_express_customers_data.csv')
df_data.head(3)

,CustomerID,Age,Income,Tenure,Education,Industry,Geographic Location,Cohort
0,1,56,52752.67735,3,Master,Technology,Europe,8/31/2023
1,2,69,55297.36435,6,Bachelor,Technology,South America,8/31/2021
2,3,46,57978.75338,3,Bachelor,Finance,Europe,5/31/2019


In [3]:
df_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1143 entries, 0 to 1142
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   CustomerID           1143 non-null   int64  
 1   Age                  1143 non-null   int64  
 2   Income               1143 non-null   float64
 3   Tenure               1143 non-null   int64  
 4   Education            1143 non-null   object 
 5   Industry             1143 non-null   object 
 6   Geographic Location  1143 non-null   object 
 7   Cohort               1143 non-null   object 
dtypes: float64(1), int64(3), object(4)
memory usage: 71.6+ KB


In [2]:
df_transactions = pd.read_csv('./resources/tlacuachitos_express_transactions.csv')
df_transactions.head(3)

,CustomerID,TransactionDate,TransactionAmount
0,1,2023-08-31,524.891753
1,1,2024-10-31,794.653366
2,1,2023-12-31,223.096087


In [4]:
df_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31552 entries, 0 to 31551
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         31552 non-null  int64  
 1   TransactionDate    31552 non-null  object 
 2   TransactionAmount  31552 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 739.6+ KB


### 2. Limpieza de datos

In [5]:
df_transactions['TransactionDate'] = pd.to_datetime(df_transactions['TransactionDate'])
df_data['Cohort'] = pd.to_datetime(df_data['Cohort'])

In [6]:
snapshot_date = df_transactions['TransactionDate'].max()
snapshot_date

Timestamp('2025-03-31 00:00:00')

In [7]:
df_transactions['TransactionDate'].min()

Timestamp('2018-01-31 00:00:00')

In [8]:
# Cantidad de meses que tien el cliente en la compañía
df_data['customer_tenure'] = (snapshot_date.year - df_data['Cohort'].dt.year)*12 + (snapshot_date.month - df_data['Cohort'].dt.month)
df_data.head(3)

,CustomerID,Age,Income,Tenure,Education,Industry,Geographic Location,Cohort,customer_tenure
0,1,56,52752.67735,3,Master,Technology,Europe,2023-08-31,19
1,2,69,55297.36435,6,Bachelor,Technology,South America,2021-08-31,43
2,3,46,57978.75338,3,Bachelor,Finance,Europe,2019-05-31,70


In [9]:
df_data.drop(columns=['Cohort'], inplace=True)
df_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1143 entries, 0 to 1142
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   CustomerID           1143 non-null   int64  
 1   Age                  1143 non-null   int64  
 2   Income               1143 non-null   float64
 3   Tenure               1143 non-null   int64  
 4   Education            1143 non-null   object 
 5   Industry             1143 non-null   object 
 6   Geographic Location  1143 non-null   object 
 7   customer_tenure      1143 non-null   int32  
dtypes: float64(1), int32(1), int64(3), object(3)
memory usage: 67.1+ KB


In [10]:
df_master = df_transactions.merge(df_data, on='CustomerID')
df_master.head(3)

,CustomerID,TransactionDate,TransactionAmount,Age,Income,Tenure,Education,Industry,Geographic Location,customer_tenure
0,1,2023-08-31,524.891753,56,52752.67735,3,Master,Technology,Europe,19
1,1,2024-10-31,794.653366,56,52752.67735,3,Master,Technology,Europe,19
2,1,2023-12-31,223.096087,56,52752.67735,3,Master,Technology,Europe,19


In [12]:
df_master['customer_tenure_on_transaction'] = (
    df_master['TransactionDate'].dt.year - df_master['customer_tenure'].dt.year)*12 + (
    df_master['TransactionDate'].dt.month - df_master['customer_tenure'].dt.month
)

df_master.head()

AttributeError: Can only use .dt accessor with datetimelike values

In [74]:
df_master[['Cohort', 'TransactionDate', 'customer_tenure_on_transaction']].head()

,Cohort,TransactionDate,customer_tenure_on_transaction
0,2023-08-31,2023-08-31,0
1,2023-08-31,2024-10-31,14
2,2023-08-31,2023-12-31,4
3,2023-08-31,2023-12-31,4
4,2023-08-31,2024-03-31,7


### 3. Cálculo del KPI objetivo: LTV a 24 meses

In [13]:
cltv_24_months = df_master[
    (df_master['customer_tenure'] > 24) &
    (df_master['customer_tenure_on_transaction'] <= 24)
].groupby('CustomerID')['TransactionAmount'].sum().reset_index()

cltv_24_months.rename(columns={'TransactionAmount': 'LTV'}, inplace=True)
cltv_24_months.head()

KeyError: 'customer_tenure_on_transaction'

### 4. Ingeniería de características

In [76]:
categorical_features = ['Education', 'Industry', 'Geographic Location']
numerical_features = ['Age', 'Income']

In [77]:
df_master['Education'].unique()

array(['Master', 'Bachelor', 'High School', 'PhD'], dtype=object)

In [78]:

df_encoded = pd.get_dummies(df_data, columns=categorical_features, drop_first=True)
df_encoded.head()

,CustomerID,Age,Income,Tenure,Cohort,customer_tenure,Education_High School,Education_Master,Education_PhD,Industry_Entertainment,Industry_Finance,Industry_Healthcare,Industry_Technology,Geographic Location_Australia,Geographic Location_Europe,Geographic Location_North America,Geographic Location_South America
0,1,56,52752.67735,3,2023-08-31,19,False,True,False,False,False,False,True,False,True,False,False
1,2,69,55297.36435,6,2021-08-31,43,False,False,False,False,False,False,True,False,False,False,True
2,3,46,57978.75338,3,2019-05-31,70,False,False,False,False,True,False,False,False,True,False,False
3,4,32,60445.26690,3,2021-02-28,49,True,False,False,False,False,False,False,False,False,False,True
4,5,60,57741.87093,5,2018-10-31,77,False,False,False,True,False,False,False,False,False,False,False


In [79]:

df_to_model = cltv_24_months.merge(df_encoded, on='CustomerID')
df_to_model.head()

,CustomerID,LTV,Age,Income,Tenure,Cohort,customer_tenure,Education_High School,Education_Master,Education_PhD,Industry_Entertainment,Industry_Finance,Industry_Healthcare,Industry_Technology,Geographic Location_Australia,Geographic Location_Europe,Geographic Location_North America,Geographic Location_South America
0,2,11276.581901,69,55297.36435,6,2021-08-31,43,False,False,False,False,False,False,True,False,False,False,True
1,3,5084.632444,46,57978.75338,3,2019-05-31,70,False,False,False,False,True,False,False,False,True,False,False
2,4,3037.917187,32,60445.26690,3,2021-02-28,49,True,False,False,False,False,False,False,False,False,False,True
3,5,11677.948404,60,57741.87093,5,2018-10-31,77,False,False,False,True,False,False,False,False,False,False,False
4,6,4556.353725,25,57132.40462,3,2022-06-30,33,False,True,False,False,False,True,False,False,False,True,False


In [80]:
dummy_features = (
    df_to_model.columns[df_to_model.columns.str.startswith(tuple(categorical_features))].values).tolist()

In [81]:
dummy_features

['Education_High School',
 'Education_Master',
 'Education_PhD',
 'Industry_Entertainment',
 'Industry_Finance',
 'Industry_Healthcare',
 'Industry_Technology',
 'Geographic Location_Australia',
 'Geographic Location_Europe',
 'Geographic Location_North America',
 'Geographic Location_South America']

### 5. Regresión lineal multivariable

In [82]:
#pip install statsmodels

In [90]:
import statsmodels.api as sm

X = df_to_model[dummy_features + numerical_features]
y = ['LTV']

X = sm.add_constant(X)

model = sm.OLS(df_to_model[y], X.astype(float))
results = model.fit()

results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    LTV   R-squared:                       0.615
Model:                            OLS   Adj. R-squared:                  0.610
Method:                 Least Squares   F-statistic:                     119.2
Date:                Thu, 03 Apr 2025   Prob (F-statistic):          1.44e-190
Time:                        16:06:38   Log-Likelihood:                -8875.8
No. Observations:                 984   AIC:                         1.778e+04
Df Residuals:                     970   BIC:                         1.785e+04
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
=====================================================================================================
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
const                             -1621.4359    352.703     -4.597      0.000   -2313.585    -929.287
Education_High School             -1539.1595    156.971     -9.805      0.000   -1847.201   -1231.118
Education_Master                   1394.9743    176.017      7.925      0.000    1049.555    1740.393
Education_PhD                      2125.8119    229.469      9.264      0.000    1675.498    2576.125
Industry_Entertainment             2689.3240    203.256     13.231      0.000    2290.452    3088.196
Industry_Finance                   1094.0132    202.207      5.410      0.000     697.200    1490.826
Industry_Healthcare                 409.9379    196.538      2.086      0.037      24.248     795.627
Industry_Technology                1956.0130    200.716      9.745      0.000    1562.124    2349.902
Geographic Location_Australia       -66.9734    204.681     -0.327      0.744    -468.643     334.696
Geographic Location_Europe          191.9400    207.613      0.925      0.355    -215.482     599.362
Geographic Location_North America   731.0676    212.564      3.439      0.001     313.930    1148.205
Geographic Location_South America   171.9923    207.475      0.829      0.407    -235.160     579.145
Age                                 128.9788      4.257     30.298      0.000     120.625     137.333
Income                                0.0323      0.004      7.744      0.000       0.024       0.040
==============================================================================
Omnibus:                       39.813   Durbin-Watson:                   2.012
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               55.753
Skew:                           0.377   Prob(JB):                     7.82e-13
Kurtosis:                       3.889   Cond. No.                     3.45e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 3.45e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

### 6. Interpretación del modelo

- ¿Qué variables son estadísticamente significativas?
- ¿Qué coeficientes son positivos o negativos?
- ¿Cómo se puede segmentar a los clientes con base en el LTV esperado?

### 7. Predicción de un nuevo cliente

In [98]:
new_customer_data = {
    'const': [1],
    'Education_High School': [0],
    'Education_Master': [0],
    'Education_PhD': [0],
    'Industry_Entertainment': [0],
    'Industry_Finance': [0],
    'Industry_Healthcare': [1],
    'Industry_Technology': [0],
    'Geographic Location_Australia': [0],
    'Geographic Location_Europe': [0],
    'Geographic Location_North America': [0],
    'Geographic Location_South America': [1],
    'Age': [18],
    'Income': [3500],
}

new_customer_df = pd.DataFrame(new_customer_data)

predicted_ltv = results.predict(new_customer_df)

print(f"Predicted LTV for the new customer: ${predicted_ltv[0]:,.2f}")


Predicted LTV for the new customer: $1,395.20


### 8. Actividad final
Caso de aplicación:
- El equipo de marketing tiene presupuesto limitado para campañas. 
- Basado en el modelo, ¿qué tipo de cliente deberías priorizar?

